In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers torch scikit-learn networkx
!pip install -q sinling sentencepiece accelerate

print(" Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.4 MB/s eta 0:00:00
 Dependencies installed


In [ ]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from transformers import AutoModel, AutoTokenizer
from sinling import SinhalaTokenizer, POSTagger
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported")

 Libraries imported


In [ ]:
# Define paths
BASE_PATH = "YOUR_PATH"
TEST_PATH = f"{BASE_PATH}/YOUR_PATH"
MODEL_PATH = f"{BASE_PATH}/YOUR_PATH"
OUTPUT_PATH = f"{BASE_PATH}/YOUR_PATH"
SUMMARY_OUTPUT = f"{OUTPUT_PATH}/YOUR_PATH"

# Create output directory
os.makedirs(SUMMARY_OUTPUT, exist_ok=True)

# Verify paths
assert os.path.exists(TEST_PATH), f"Test path not found: {TEST_PATH}"
assert os.path.exists(MODEL_PATH), f"Model path not found: {MODEL_PATH}"

print(" Paths configured:")
print(f"   Test docs: {TEST_PATH}")
print(f"   Model: {MODEL_PATH}")
print(f"   Output: {SUMMARY_OUTPUT}")

In [ ]:
print("Initializing tools...")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Sinhala tools
tokenizer_sin = SinhalaTokenizer()
pos_tagger = POSTagger()

# mBERT tokenizer
tokenizer_mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

print(" Tools initialized")

Initializing tools...
Using device: cuda


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

 Tools initialized


In [ ]:
# Legal keywords for weighted graphs
LEGAL_KEYWORDS = [
    'අධිකරණය',
    'නීතිය',
    'විනිශ්චය',
    'දඬුවම',
    'චෝදනා',
    'සාක්ෂි',
    'තීන්දුව',
    'අභියාචනය',
    'නඩුව',
    'වරද',
    'දෝෂ',
    'බලය',
    'කොන්දේසි',
    'ප්‍රතිපාදන',
    'ව්‍යවස්ථා',
    'පනත',
    'පැමිණිලිකරු',
    'කොන්ත්‍රාත්තුව',
    'පැමිණිල්ල'
]

print(f" Loaded {len(LEGAL_KEYWORDS)} legal keywords")

 Loaded 19 legal keywords


In [ ]:
print("Loading mBERT for sentence embeddings...")

model_mbert = AutoModel.from_pretrained("bert-base-multilingual-cased")
model_mbert.to(device)
model_mbert.eval()

def get_sentence_embedding(sentence):
    """Generate sentence embedding using mBERT"""
    inputs = tokenizer_mbert(sentence, return_tensors="pt",
                             max_length=512, truncation=True,
                             padding=True).to(device)
    with torch.no_grad():
        outputs = model_mbert(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(" mBERT loaded for embeddings")

Loading mBERT for sentence embeddings...


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

 mBERT loaded for embeddings


In [ ]:
def cluster_sentences(sentences, num_clusters=3):
    """
    Cluster sentences by semantic similarity
    """
    if len(sentences) <= num_clusters:
        return [[sent] for sent in sentences]

    # Generate embeddings
    print(f"   Generating embeddings for {len(sentences)} sentences...")
    embeddings = np.array([get_sentence_embedding(sent) for sent in sentences])

    # Cluster
    clustering = AgglomerativeClustering(
        n_clusters=min(num_clusters, len(sentences)),
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)

    # Group by cluster
    clusters = {}
    for idx, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(sentences[idx])

    return list(clusters.values())

print(" Clustering function defined")

 Clustering function defined


In [ ]:
KEEP_POS = {'NOUN', 'VERB', 'AUX', 'PROPN', 'ADJ'}

def build_word_graph_weighted(sentences_in_cluster, legal_keywords):
    """
    Build word graph with LEGAL KEYWORD-AWARE EDGE WEIGHTS
    FIXED: More robust - doesn't rely solely on POS tagging
    """
    G = nx.DiGraph()
    G.add_node("START")
    G.add_node("END")

    total_words_added = 0

    for sentence in sentences_in_cluster:
        tokenized = tokenizer_sin.tokenize(sentence)
        if len(tokenized) == 0:
            continue

        # Try POS tagging, but don't fail if it doesn't work
        words_with_pos = []

        try:
            pos_tags = pos_tagger.predict([tokenized])[0]
            words_with_pos = [(w, p) for w, p in zip(tokenized, pos_tags)
                             if p in KEEP_POS and len(w) > 1]
        except:
            pass

        # FALLBACK: If POS tagging failed or gave too few words,
        # use ALL words (filtering only by length and stopwords)
        if len(words_with_pos) < 3:
            # Simple heuristic: keep words longer than 2 characters
            # and prioritize words with legal keywords
            words_with_pos = []
            for w in tokenized:
                if len(w) > 2:  # Skip very short words
                    # Assign dummy POS tag
                    words_with_pos.append((w, 'NOUN'))

        # Limit words per sentence to avoid too dense graphs
        words_with_pos = words_with_pos[:30]

        if len(words_with_pos) == 0:
            continue

        # START → first word
        first_word = words_with_pos[0]
        weight = 2.0 if any(kw in first_word[0] for kw in legal_keywords) else 1.0

        if G.has_edge("START", first_word):
            G["START"][first_word]['weight'] += weight
        else:
            G.add_edge("START", first_word, weight=weight)

        total_words_added += 1

        # Consecutive words
        for i in range(len(words_with_pos)-1):
            current = words_with_pos[i]
            next_word = words_with_pos[i+1]

            # Higher weight if either word is legal keyword
            has_keyword = any(kw in current[0] or kw in next_word[0]
                            for kw in legal_keywords)
            weight = 2.0 if has_keyword else 1.0

            if G.has_edge(current, next_word):
                G[current][next_word]['weight'] += weight
            else:
                G.add_edge(current, next_word, weight=weight)

            total_words_added += 1

        # last → END
        last_word = words_with_pos[-1]
        weight = 2.0 if any(kw in last_word[0] for kw in legal_keywords) else 1.0

        if G.has_edge(last_word, "END"):
            G[last_word]["END"]['weight'] += weight
        else:
            G.add_edge(last_word, "END", weight=weight)

    print(f"       Built graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    return G

print(" FIXED keyword-weighted graph building function defined")

 FIXED keyword-weighted graph building function defined


In [ ]:
import random

def generate_fused_sentences(G, num_paths=5, max_length=25, min_length=8):
    """
    Generate high-quality abstractive candidates
    IMPROVED: Better coherence, length control, deduplication
    """
    if G.number_of_nodes() <= 2:
        return []

    candidates = []
    seen_word_sets = set()  # Avoid duplicate word combinations

    # Method 1: Weighted random walks (favor high-weight edges)
    for attempt in range(num_paths * 5):  # More attempts for better candidates
        try:
            path = ["START"]
            current = "START"
            words = []

            # Random walk with length constraints
            for step in range(max_length + 5):  # Allow some overshoot
                neighbors = list(G.successors(current))

                # Remove END from neighbors unless we have enough words
                if len(words) < min_length and "END" in neighbors:
                    neighbors.remove("END")

                if len(neighbors) == 0:
                    break

                # If we have enough words and END is available, maybe take it
                if len(words) >= min_length and "END" in neighbors:
                    if random.random() < 0.3:  # 30% chance to end
                        break

                # Weighted selection
                weights = []
                valid_neighbors = []
                for n in neighbors:
                    if n != "END":
                        weight = G[current][n].get('weight', 1.0)
                        weights.append(weight)
                        valid_neighbors.append(n)

                if len(valid_neighbors) == 0:
                    break

                # Pick next node with weighted probability
                next_node = random.choices(valid_neighbors, weights=weights)[0]

                # Extract word
                if isinstance(next_node, tuple):
                    word = next_node[0]
                    if len(word) > 1:  # Skip single chars
                        words.append(word)

                current = next_node

                # Stop if we reach max length
                if len(words) >= max_length:
                    break

            # Check if we have valid length
            if len(words) < min_length or len(words) > max_length:
                continue

            # Check for duplicates
            word_set = frozenset(words)
            if word_set in seen_word_sets:
                continue

            seen_word_sets.add(word_set)

            # Create sentence
            fused = ' '.join(words)

            # Quality checks
            if len(fused) < 20:  # Too short
                continue

            # Check for excessive repetition
            unique_words = len(set(words))
            if unique_words < len(words) * 0.6:  # More than 40% repetition
                continue

            candidates.append(fused)

            if len(candidates) >= num_paths:
                break

        except Exception as e:
            continue

    # Method 2: If we don't have enough, use weighted word extraction
    if len(candidates) < num_paths:
        # Get words sorted by importance (edge weight)
        word_weights = {}

        for u, v, data in G.edges(data=True):
            for node in [u, v]:
                if isinstance(node, tuple) and node not in ["START", "END"]:
                    word = node[0]
                    if len(word) > 1:
                        weight = data.get('weight', 1.0)
                        word_weights[word] = word_weights.get(word, 0) + weight

        if len(word_weights) >= min_length:
            # Sort by weight
            sorted_words = sorted(word_weights.items(), key=lambda x: x[1], reverse=True)

            # Create different combinations
            attempts = 0
            while len(candidates) < num_paths and attempts < 10:
                attempts += 1

                # Take top weighted words with some randomness
                n_words = random.randint(min_length, min(max_length, len(sorted_words)))

                if attempts == 1:
                    # First attempt: pure top words
                    selected = [w for w, _ in sorted_words[:n_words]]
                else:
                    # Later attempts: mix high and medium weight words
                    top_k = len(sorted_words) // 2
                    top_words = [w for w, _ in sorted_words[:top_k]]
                    selected = random.sample(top_words, min(n_words, len(top_words)))

                # Check uniqueness
                word_set = frozenset(selected)
                if word_set in seen_word_sets:
                    continue

                seen_word_sets.add(word_set)

                fused = ' '.join(selected)

                # Quality check
                if len(fused) >= 20:
                    candidates.append(fused)

    return candidates[:num_paths]  # Return at most num_paths

print("Path generation function defined")

Path generation function defined


In [ ]:
def filter_low_quality_candidates(candidates):
    """
    Filter out obviously bad candidates
    """
    filtered = []

    for candidate in candidates:
        # Skip if too short
        if len(candidate) < 20:
            continue

        # Skip if too many repeated words
        words = candidate.split()
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.6:  # More than 40% repetition
            continue

        # Skip if same word appears 3+ times
        from collections import Counter
        word_counts = Counter(words)
        max_count = max(word_counts.values())
        if max_count >= 3:
            continue

        # All checks passed
        filtered.append(candidate)

    return filtered if len(filtered) > 0 else candidates  # Return original if all filtered

print(" Quality filtering function defined")

 Quality filtering function defined


In [ ]:
print("Loading trained sentence scorer...")

# Define model architecture
class LegalSentenceScorer(nn.Module):
    def __init__(self, model_name="bert-base-multilingual-cased", dropout=0.3):
        super(LegalSentenceScorer, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        score = self.classifier(cls_embedding)
        return score.squeeze()

# Initialize and load trained model
scorer_model = LegalSentenceScorer()
checkpoint = torch.load(f"{MODEL_PATH}/legal_sentence_scorer.pt", map_location=device)
scorer_model.load_state_dict(checkpoint['model_state_dict'])
scorer_model.to(device)
scorer_model.eval()

print(f" Scorer model loaded (Val Accuracy: {checkpoint['best_val_accuracy']:.4f})")

Loading trained sentence scorer...
 Scorer model loaded (Val Accuracy: 0.8363)


In [ ]:
def generate_summary_with_scoring(document, num_clusters=3):
    """
    Generate abstractive summary with improved quality
    IMPROVEMENTS: Longer output, multiple sentences per cluster, complete endings
    """
    # Split into sentences
    import re
    split_pattern = re.compile(r'[.!?;]+')
    sentences = [s.strip() for s in split_pattern.split(document)
                if len(s.strip()) > 20]

    if len(sentences) < 3:
        return '. '.join(sentences) + '.'

    # Step 1: Cluster sentences
    print(f"   Clustering {len(sentences)} sentences...")
    clusters = cluster_sentences(sentences, num_clusters=min(3, max(2, len(sentences)//5)))
    print(f"   Created {len(clusters)} clusters")

    summary_sentences = []

    # Step 2: Process each cluster - SELECT TOP 2 CANDIDATES
    for cluster_idx, cluster in enumerate(clusters, 1):
        if len(cluster) == 0:
            continue

        print(f"   Processing cluster {cluster_idx}/{len(clusters)} ({len(cluster)} sentences)...")

        # Build keyword-weighted graph
        G = build_word_graph_weighted(cluster, LEGAL_KEYWORDS)

        if G.number_of_nodes() <= 10:
            print(f"       Graph has only {G.number_of_nodes()} nodes, skipping")
            continue

        # Generate MORE candidates with LONGER length
        candidates = generate_fused_sentences(G, num_paths=8, max_length=25, min_length=8)

        if len(candidates) == 0:
            print(f"      No candidates generated, skipping cluster")
            continue

        # Filter low quality
        candidates = filter_low_quality_candidates(candidates)

        if len(candidates) == 0:
            print(f"       All candidates filtered out, skipping cluster")
            continue

        print(f"     Generated {len(candidates)} abstractive candidates, scoring...")

        # Score ALL candidates
        candidate_scores = []

        for cand_idx, candidate in enumerate(candidates):
            encoding = tokenizer_mbert(
                candidate,
                add_special_tokens=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            with torch.no_grad():
                score = scorer_model(
                    encoding['input_ids'].to(device),
                    encoding['attention_mask'].to(device)
                ).item()

            candidate_scores.append((candidate, score))
            print(f"       Candidate {cand_idx+1}: {score:.4f} | {candidate[:60]}...")

        # Sort by score
        candidate_scores.sort(key=lambda x: x[1], reverse=True)

        # CHANGE: Take TOP 2 candidates per cluster
        top_k = min(2, len(candidate_scores))

        for i in range(top_k):
            candidate, score = candidate_scores[i]
            print(f"     Selected #{i+1} (score: {score:.4f})")
            summary_sentences.append(candidate)

    # Step 3: Combine with smart formatting
    if len(summary_sentences) == 0:
        return '. '.join(sentences[:3]) + '.'

    # Join all selected sentences
    summary = '. '.join(summary_sentences)

    # Step 4: Ensure proper sentence ending
    # Target: 400-500 characters
    target_min = 300
    target_max = 500

    if len(summary) < target_min:
        # Too short - add period
        if not summary.endswith('.'):
            summary += '.'

    elif len(summary) <= target_max:
        # Perfect length - just ensure period
        if not summary.endswith('.'):
            summary += '.'

    else:
        # Too long - truncate at sentence boundary

        # Try to find last complete sentence before target_max
        sentences_in_summary = [s.strip() for s in summary.split('.') if s.strip()]

        # Build summary by adding sentences until we exceed target
        final_summary = ""
        for sent in sentences_in_summary:
            test_summary = final_summary + sent + '. '

            if len(test_summary) > target_max:
                # This sentence would make it too long
                if len(final_summary) > target_min:
                    # We already have enough content, stop here
                    break
                else:
                    # We need this sentence even if it's a bit long
                    final_summary = test_summary
                    break
            else:
                final_summary = test_summary

        summary = final_summary.strip()

        # Ensure ends with period
        if not summary.endswith('.'):
            # Find last complete word
            last_space = summary.rfind(' ')
            if last_space > target_min:
                summary = summary[:last_space] + '.'
            else:
                summary += '.'

    # Final safety: always end with period
    if not summary.endswith('.'):
        summary += '.'

    return summary

print("Summary generation with longer output")

Summary generation with longer output


In [ ]:
print("Loading test documents...")

test_files = [f for f in os.listdir(TEST_PATH) if f.endswith('.txt')]
print(f" Found {len(test_files)} test documents")

Loading test documents...
 Found 242 test documents


In [ ]:
print("\n" + "="*60)
print("GENERATING SUMMARIES")
print("="*60 + "\n")

successful = 0
failed = 0
failed_files = []

for i, filename in enumerate(test_files, 1):
    print(f"\n[{i}/{len(test_files)}] Processing: {filename}")
    print("-" * 60)

    try:
        # Load document
        filepath = os.path.join(TEST_PATH, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            document = f.read().strip()

        if len(document) < 100:
            print("     Document too short, skipping")
            failed += 1
            failed_files.append(filename)
            continue

        # Generate summary
        summary = generate_summary_with_scoring(document, num_clusters=3)

        # Save summary
        base_name = os.path.splitext(filename)[0]
        output_file = os.path.join(SUMMARY_OUTPUT, f"{base_name}_summary.txt")

        with open(output_file, 'w', encoding='utf-8') as out:
            out.write(summary)

        print(f"\n    Summary generated ({len(summary)} chars)")
        print(f"    Preview: {summary[:150]}...")

        successful += 1

    except Exception as e:
        print(f"    Error: {e}")
        failed += 1
        failed_files.append(filename)

print("\n" + "="*60)
print("SUMMARY GENERATION COMPLETE")
print("="*60)
print(f"\n Successfully generated: {successful} summaries")
print(f" Failed: {failed} documents")

if failed_files:
    print(f"\n  Failed files:")
    for f in failed_files[:10]:  # Show first 10
        print(f"   - {f}")

print(f"\n Summaries saved to: {SUMMARY_OUTPUT}")

Streaming output truncated to the last 5000 lines.

[115/242] Processing: acts_1986-07-24_Land_Reform_Special_Provisions_Amendment_si.txt
------------------------------------------------------------
   Clustering 10 sentences...
   Generating embeddings for 10 sentences...
   Created 2 clusters
   Processing cluster 1/2 (9 sentences)...
       Built graph: 129 nodes, 185 edges
     Generated 7 abstractive candidates, scoring...
       Candidate 1: 0.0094 | පනත 1981 ජූනි දින ආණ්ඩුවේ නියමය පරිදි කොමිෂන් සභාව විසින් දෙ...
       Candidate 2: 0.0272 | iii ඡේදයේ කාර්ය සඳහා වන්දිය ලැබිය යුතු අතර අනුව වන්දිය ලැබිය...
       Candidate 3: 0.0095 | පනත 1986 අංක දරන ඉඩම් ප්‍රතිසංස්කරණ විශේෂ විධිවිධාන පනතේ වගන...
       Candidate 4: 0.3927 | මෙහි ඉහතින් විධිවිධාන පනත සංශෝධනය සියලු කාර්ය සඳහා මුදල හැර ...
       Candidate 5: 0.0095 | උපවගන්තියෙහි සඳහන් එවැනි යම් සමාගමක කොටස් පන්තියට සම්බන්ධ අය...
       Candidate 6: 0.0095 | 1986 ජූලි වැනි දින ආණ්ඩුවේ නියමය පරිදි කොමිෂන් සභාව විසින් ද...
       Can

In [ ]:
print("\n" + "="*60)
print("TESTING ON SAMPLE DOCUMENT")
print("="*60)

# Pick a random test document
import random
sample_file = random.choice(test_files)
sample_path = os.path.join(TEST_PATH, sample_file)

print(f"\nSample document: {sample_file}")

with open(sample_path, 'r', encoding='utf-8') as f:
    sample_doc = f.read()

print(f"\nOriginal document length: {len(sample_doc)} characters")
print(f"Original document preview:\n{sample_doc[:300]}...\n")

# Generate summary
print("Generating summary...")
sample_summary = generate_summary_with_scoring(sample_doc, num_clusters=3)

print(f"\n{'='*60}")
print("GENERATED SUMMARY:")
print(f"{'='*60}")
print(sample_summary)
print(f"\n Summary length: {len(sample_summary)} characters")


TESTING ON SAMPLE DOCUMENT

Sample document: acts_2000-07-19_Katuwana_Adults_Care_Centre_Development_Foundation_Incorporation_si.txt

Original document length: 7677 characters
Original document preview:
ශ්‍රී ලංකා ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
2000 අංක 29 දරන කටුවන වැඩිහිටි නිවාස සංවර්ධන
පදනම (සංස්ථාගත කිරීමේ) පනත

(සහතිකය සටහන් කළේ 2000 ජුලි මස 19 වන දින.]
ආණ්ඩුවේ නියමය පරිදි මුද්‍රණය කරන ලදී.

2000 ජූලි මස 21 වැනි දින ශ්‍රී ලංකා ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ ගැසට්
පත්‍රයේ II ව...

Generating summary...
   Clustering 36 sentences...
   Generating embeddings for 36 sentences...
   Created 3 clusters
   Processing cluster 1/3 (29 sentences)...
       Built graph: 357 nodes, 571 edges
     Generated 7 abstractive candidates, scoring...
       Candidate 1: 0.2818 | සංස්ථාවේ ගිණුම් ආරම්භ කිරීම සම්බන්ධීකරණය කිරීම සඳහා පනතේ වෙන...
       Candidate 2: 0.0094 | 2000 ජූලි වැනි දින සිට ඉන් පසුව මෙහි දක්වා ඇති පරිදි සංස්ථාව...
       Candidate 3: 0.0093 | පනත සහතිකය සටහන් කළේ

In [ ]:
# Save statistics
stats = {
    'total_test_documents': len(test_files),
    'successful_summaries': successful,
    'failed_documents': failed,
    'success_rate': successful / len(test_files) * 100 if len(test_files) > 0 else 0,
    'output_directory': SUMMARY_OUTPUT,
    'modifications': [
        'Legal keyword-weighted graph construction',
        'Multi-path candidate generation with neural scoring'
    ],
    'parameters': {
        'num_clusters': 3,
        'num_paths_per_cluster': 5,
        'max_sentence_length': 20,
        'summary_max_length': 300
    }
}

stats_file = f"{OUTPUT_PATH}/summary_generation_stats.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"\n Statistics saved to {stats_file}")